## atachment

In [11]:
from langchain_openai import ChatOpenAI 
from langchain_core.messages import AIMessage 
from langgraph.graph import MessagesState
from typing import List
from langgraph.types import Command 
from backend.chatbot.prompts.attachment import prompt as attachment_prompt
from backend.chatbot.prompts.schema import prompt as  schema_prompt 
from langchain_core.messages import SystemMessage, HumanMessage , AIMessage
from typing import List, Dict, Any, Literal
from typing_extensions import Annotated

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key='sk-1666caf6a268456d8df5b5da9853d5d6',
    base_url="https://api.deepseek.com",
    temperature=0.7,
)

def merge_dicts(old, new): # i used this to let collected output gest updated before overwriting
    return {**old, **new}

class GraphState(MessagesState):
    remaining_agents: List[str]
    collected_outputs: Annotated[Dict[str, Any], merge_dicts]


def create_node(llm  , prompt , node_name):
    def agent(state:GraphState): 
        user_context = state["messages"][-1].content
        messages = [
            SystemMessage(content= prompt), 
            HumanMessage(content = user_context)
        ]

        response = llm.invoke(messages)
        # updated_messages = state['messages'] + [
        #     AIMessage(content = response.content , name = node_name)
        # ]
        import json 
        data = json.loads(response.content)

        return Command(
            update = {
                  'collected_outputs':  data
            }, 
            goto = 'supervisor'
        )
    return agent

attachment_agent = create_node(llm, attachment_prompt , node_name = 'atthchment_agent')
schema_agent = create_node(llm, schema_prompt , node_name = 'schema_agent')
schema_agent

<function __main__.create_node.<locals>.agent(state: __main__.GraphState)>

In [ ]:
from user_context import first 

state = {
    'messages' : [
        HumanMessage(content=first)
    ]
}
response = attachment_agent(state)
result = schema_agent(state)

Command(update={'collected_outputs': {'attachment_style': {'Secure': 1, 'Anxious-Preoccupied': 9, 'Dismissive-Avoidant': 4, 'Fearful-Avoidant': 7}, 'summary': "The person is deeply conflicted about a past relationship, oscillating between intense longing and a strong need for independence (Anxious-Preoccupied 9). They experience guilt and shame over a fleeting memory of an ex while with a current partner, and feel trapped by the other's attempts to maintain contact, yet they also admit to hoping for change in the future (Fearful-Avoidant 7). They express a desire to avoid emotional entanglement and assert boundaries ('nemiikham bebinamet'), but simultaneously feel drawn back into the cycle (Dismissive-Avoidant 4). There is little evidence of secure attachment; they report never feeling safe or heard in the relationship (Secure 1)."}}, goto='supervisor')


In [15]:
state = {
    "messages": [HumanMessage(content=first)],
    "collected_outputs": {}
}

cmd1 = attachment_agent(state)

state["collected_outputs"] = merge_dicts(
    state["collected_outputs"],
    cmd1.update["collected_outputs"]
)

cmd2 = schema_agent(state)

state["collected_outputs"] = merge_dicts(
    state["collected_outputs"],
    cmd2.update["collected_outputs"]
)

print(state["collected_outputs"])


{'attachment_style': {'Secure': 1, 'Anxious-Preoccupied': 9, 'Dismissive-Avoidant': 5, 'Fearful-Avoidant': 8}, 'summary': 'شخص درگیر رابطه\u200cای است که در آن احساس بی\u200cثباتی و ترس از رها شدن دارد (Abandonment/Instability 8). او به طرف مقابل اعتماد ندارد و احساس می\u200cکند مورد سوءاستفاده و تحقیر قرار می\u200cگیرد (Mistrust/Abuse 7). همچنین از نظر عاطفی احساس محرومیت می\u200cکند (Emotional Deprivation 6) و عمیقاً احساس نقص و شرم می\u200cکند (Defectiveness/Shame 9). او خود را جدا از دیگران و منزوی می\u200cبیند (Social Isolation/Alienation 5). وابستگی عاطفی ناایمن و احساس درماندگی در تصمیم\u200cگیری دارد (Dependence/Incompetence 3). احساس می\u200cکند هویت خود را در رابطه گم کرده و با دیگری ادغام شده است (Enmeshment/Undeveloped Self 7). او خود را در رسیدن به خواسته\u200cهایش ناکام می\u200cبیند (Failure 6) و به دنبال تأیید دیگران است (Approval-Seeking/Recognition-Seeking 5). نگاه او به آینده بسیار بدبینانه و منفی است (Negativity/Pessimism 8). استانداردهای سخت\u200cگیرانه\u200cای برای

In [17]:
state['collected_outputs']

{'attachment_style': {'Secure': 1,
  'Anxious-Preoccupied': 9,
  'Dismissive-Avoidant': 5,
  'Fearful-Avoidant': 8},
 'summary': 'شخص درگیر رابطه\u200cای است که در آن احساس بی\u200cثباتی و ترس از رها شدن دارد (Abandonment/Instability 8). او به طرف مقابل اعتماد ندارد و احساس می\u200cکند مورد سوءاستفاده و تحقیر قرار می\u200cگیرد (Mistrust/Abuse 7). همچنین از نظر عاطفی احساس محرومیت می\u200cکند (Emotional Deprivation 6) و عمیقاً احساس نقص و شرم می\u200cکند (Defectiveness/Shame 9). او خود را جدا از دیگران و منزوی می\u200cبیند (Social Isolation/Alienation 5). وابستگی عاطفی ناایمن و احساس درماندگی در تصمیم\u200cگیری دارد (Dependence/Incompetence 3). احساس می\u200cکند هویت خود را در رابطه گم کرده و با دیگری ادغام شده است (Enmeshment/Undeveloped Self 7). او خود را در رسیدن به خواسته\u200cهایش ناکام می\u200cبیند (Failure 6) و به دنبال تأیید دیگران است (Approval-Seeking/Recognition-Seeking 5). نگاه او به آینده بسیار بدبینانه و منفی است (Negativity/Pessimism 8). استانداردهای سخت\u200cگیرانه\u200c

In [ ]:
import json
output_content = response.update['collected_outputs']
output_content

{'attachment_style': {'Secure': 1,
  'Anxious-Preoccupied': 9,
  'Dismissive-Avoidant': 6,
  'Fearful-Avoidant': 8},
 'summary': 'The person is caught in a turbulent cycle of longing and rejection, expressing intense anxiety about the relationship and a deep fear of being unworthy or unlovable (Anxious-Preoccupied 9). They oscillate between wanting closeness and feeling suffocated, often withdrawing to protect themselves (Dismissive-Avoidant 6), while simultaneously experiencing guilt, confusion, and a desire for freedom that conflicts with their attachment (Fearful-Avoidant 8). There is minimal evidence of secure, stable attachment (Secure 1). The text reveals a painful pattern of seeking validation from a partner who triggers insecurity, and the person struggles with contradictory feelings of hope and despair.'}

In [32]:
import json


output_content = result.update['messages'][1].content
json_output = json.loads(output_content)
json_output

{'schemas': {'Abandonment/Instability': 7,
  'Mistrust/Abuse': 8,
  'Emotional Deprivation': 9,
  'Defectiveness/Shame': 8,
  'Social Isolation/Alienation': 5,
  'Dependence/Incompetence': 4,
  'Vulnerability to Harm or Illness': 3,
  'Enmeshment/Undeveloped Self': 6,
  'Failure': 5,
  'Entitlement/Grandiosity': 0,
  'Insufficient Self-Control/Self-Discipline': 2,
  'Subjugation': 7,
  'Self-Sacrifice': 3,
  'Approval-Seeking/Recognition-Seeking': 4,
  'Negativity/Pessimism': 8,
  'Emotional Inhibition': 6,
  'Unrelenting Standards/Hypercriticalness': 4,
  'Punitiveness': 5},
 'summary': 'فرد درگیر احساسات شدید گناه و سردرگمی پس از ملاقات با شریک سابقش است. او از اینکه بوی گردن شریکش او را به یاد شخص دیگری انداخته، احساس شرم و نقص میکند (Defectiveness/Shame 8) و خود را به بی\u200cتعهدی اخلاقی متهم می\u200cکند. رابطه گذشته را پر از بی\u200cثباتی، ترک\u200cشدگی احساسی و بازی\u200cهای روانی توصیف می\u200cکند (Abandonment/Instability 7) و عمیقاً احساس می\u200cکند که نیازهای عاطفی او هرگز ب